# V3 vs V4 Dataset Comparison
## Side-by-side feature audit — what's broken, what's alive, what changed

V3 (batched builder, Feb 3 2026) vs V4 (live extractor, Mar 21 2026)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

# Use float32 to halve memory footprint
v3 = pd.read_csv('../nba/features/datasets/xl_training_POINTS_2023_2025_batched.csv')
v4 = pd.read_csv('../nba/features/datasets/xl_training_POINTS_2023_2025.csv')

# Downcast numeric columns to float32
for df in [v3, v4]:
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')

print(f'V3: {v3.shape[0]:,} rows x {v3.shape[1]} cols  |  Date: {v3["game_date"].min()} to {v3["game_date"].max()}')
print(f'V4: {v4.shape[0]:,} rows x {v4.shape[1]} cols  |  Date: {v4["game_date"].min()} to {v4["game_date"].max()}')
print(f'V3 players: {v3["player_name"].nunique()}  |  V4 players: {v4["player_name"].nunique()}')
print(f'V3 is_home dtype: {v3["is_home"].dtype}  |  V4 is_home dtype: {v4["is_home"].dtype}')
print(f'Memory: V3={v3.memory_usage(deep=True).sum()/1e6:.0f}MB  V4={v4.memory_usage(deep=True).sum()/1e6:.0f}MB')

## 1. Dead Feature Comparison

In [ ]:
meta = ['player_name','game_date','stat_type','source','actual_points','label','split','opponent_team','is_home','line']

def feature_report(df, name):
    feat_cols = [c for c in df.columns if c not in meta]
    rows = []
    for c in sorted(feat_cols):
        s = df[c]
        if not np.issubdtype(s.dtype, np.number):
            continue
        rows.append({
            'feature': c,
            'mean': round(s.mean(), 4),
            'std': round(s.std(), 4),
            'min': round(s.min(), 2),
            'max': round(s.max(), 2),
            'nonzero_pct': round((s != 0).mean() * 100, 1),
            'dead': s.std() == 0,
        })
    return pd.DataFrame(rows)

r3 = feature_report(v3, 'V3')
r4 = feature_report(v4, 'V4')

print(f'V3: {len(r3)} numeric features, {r3["dead"].sum()} dead')
print(f'V4: {len(r4)} numeric features, {r4["dead"].sum()} dead')
print()

# V4 dead features
print('=== V4 DEAD FEATURES ===')
v4_dead = r4[r4['dead']][['feature','mean','min','max']]
for _, row in v4_dead.iterrows():
    # Check if same feature was alive in V3
    v3_match = r3[r3['feature'] == row['feature']]
    if len(v3_match) > 0 and not v3_match.iloc[0]['dead']:
        v3_row = v3_match.iloc[0]
        print(f"  REGRESSION  {row['feature']}: V4={row['mean']} (const) | V3 range [{v3_row['min']}, {v3_row['max']}] std={v3_row['std']}")
    elif len(v3_match) > 0:
        print(f"  BOTH DEAD   {row['feature']}: V4={row['mean']}")
    else:
        print(f"  V4 ONLY     {row['feature']}: {row['mean']} (not in V3)")

## 2. Shared Features — Side-by-Side Stats

In [ ]:
# Merge on feature name
merged = r3.merge(r4, on='feature', suffixes=('_v3', '_v4'), how='outer')

# Shared features
shared = merged.dropna(subset=['mean_v3', 'mean_v4'])
print(f'Shared features: {len(shared)}')
print(f'V3-only features: {merged["mean_v4"].isna().sum()}')
print(f'V4-only features: {merged["mean_v3"].isna().sum()}')
print()

# Show V3-only
v3_only = merged[merged['mean_v4'].isna()][['feature','mean_v3','std_v3','nonzero_pct_v3']]
if len(v3_only) > 0:
    print('=== V3-ONLY FEATURES (missing from V4) ===')
    print(v3_only.to_string(index=False))
    print()

# Show V4-only
v4_only = merged[merged['mean_v3'].isna()][['feature','mean_v4','std_v4','nonzero_pct_v4','dead_v4']]
if len(v4_only) > 0:
    print('=== V4-ONLY FEATURES (new in V4) ===')
    print(v4_only.to_string(index=False))

## 3. Feature Regressions — Same Feature, Different Values

In [ ]:
# For shared features: compare distributions
comparison = []
for _, row in shared.iterrows():
    f = row['feature']
    mean_diff = abs(row['mean_v4'] - row['mean_v3'])
    std_ratio = row['std_v4'] / row['std_v3'] if row['std_v3'] > 0 else (0 if row['std_v4'] == 0 else 999)
    
    # Flag regressions
    regression = ''
    if row['dead_v4'] and not row['dead_v3']:
        regression = 'DEAD IN V4'
    elif std_ratio < 0.1 and row['std_v3'] > 0.01:
        regression = 'VARIANCE COLLAPSED'
    elif std_ratio > 10 and row['std_v3'] > 0.01:
        regression = 'VARIANCE EXPLODED'
    elif mean_diff > 5 * max(row['std_v3'], 0.01):
        regression = 'MEAN SHIFTED'
    
    comparison.append({
        'feature': f,
        'mean_v3': row['mean_v3'],
        'mean_v4': row['mean_v4'],
        'std_v3': row['std_v3'],
        'std_v4': row['std_v4'],
        'std_ratio': round(std_ratio, 3),
        'nz_v3': row['nonzero_pct_v3'],
        'nz_v4': row['nonzero_pct_v4'],
        'regression': regression,
    })

cdf = pd.DataFrame(comparison)
regressions = cdf[cdf['regression'] != ''].sort_values('regression')
print(f'Regressions found: {len(regressions)}')
print()
regressions[['feature','mean_v3','mean_v4','std_v3','std_v4','nz_v3','nz_v4','regression']]

## 4. Correlation with Label — V3 vs V4

In [ ]:
# Correlation with label for shared features
corr_comparison = []
for f in shared['feature']:
    if f in v3.columns and f in v4.columns:
        s3 = v3[f]
        s4 = v4[f]
        if np.issubdtype(s3.dtype, np.number) and s3.std() > 0:
            c3 = s3.corr(v3['label'])
        else:
            c3 = 0
        if np.issubdtype(s4.dtype, np.number) and s4.std() > 0:
            c4 = s4.corr(v4['label'])
        else:
            c4 = 0
        corr_comparison.append({'feature': f, 'corr_v3': round(c3, 4), 'corr_v4': round(c4, 4), 'diff': round(c4 - c3, 4)})

ccdf = pd.DataFrame(corr_comparison)

# Biggest correlation drops
print('=== BIGGEST CORRELATION DROPS (V3 → V4) ===')
drops = ccdf.sort_values('diff').head(20)
print(drops.to_string(index=False))
print()

# Biggest correlation gains
print('=== BIGGEST CORRELATION GAINS (V3 → V4) ===')
gains = ccdf.sort_values('diff', ascending=False).head(20)
print(gains.to_string(index=False))

In [ ]:
# Scatter plot: V3 corr vs V4 corr
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(ccdf['corr_v3'], ccdf['corr_v4'], alpha=0.5, s=20)
ax.plot([-0.2, 0.2], [-0.2, 0.2], 'r--', alpha=0.5, label='y=x (no change)')
ax.set_xlabel('V3 correlation with label')
ax.set_ylabel('V4 correlation with label')
ax.set_title('Feature-Label Correlations: V3 vs V4')

# Annotate biggest movers
for _, row in ccdf.sort_values('diff').head(5).iterrows():
    ax.annotate(row['feature'], (row['corr_v3'], row['corr_v4']), fontsize=7, alpha=0.8)
for _, row in ccdf.sort_values('diff', ascending=False).head(5).iterrows():
    ax.annotate(row['feature'], (row['corr_v3'], row['corr_v4']), fontsize=7, alpha=0.8)

ax.legend()
plt.tight_layout()
plt.show()

## 5. H2H Features Deep Dive

In [ ]:
h2h_feats = [c for c in shared['feature'] if c.startswith('h2h_')]
print(f'Shared H2H features: {len(h2h_feats)}')
print()

for f in sorted(h2h_feats):
    s3 = v3[f] if f in v3.columns else None
    s4 = v4[f] if f in v4.columns else None
    if s3 is not None and s4 is not None:
        c3 = s3.corr(v3['label']) if s3.std() > 0 else 0
        c4 = s4.corr(v4['label']) if s4.std() > 0 else 0
        pop3 = (s3 != 0).mean() * 100
        pop4 = (s4 != 0).mean() * 100
        flag = '  <<<' if abs(c3 - c4) > 0.03 else ''
        print(f'{f:<35} V3: mean={s3.mean():>8.2f} std={s3.std():>6.2f} pop={pop3:>5.1f}% corr={c3:>+.4f}  |  V4: mean={s4.mean():>8.2f} std={s4.std():>6.2f} pop={pop4:>5.1f}% corr={c4:>+.4f}{flag}')

## 6. Team/Pace Features Deep Dive

In [ ]:
pace_feats = ['team_pace','opponent_pace','pace_diff','projected_possessions',
              'team_off_rating','team_def_rating','opponent_def_rating',
              'opp_def_factor','opp_positional_def','position_matchup_advantage',
              'game_velocity','expected_possessions','projected_team_possessions',
              'game_pace','opp_score_margin_avg']

print(f'{"Feature":<30} {"V3 mean":>10} {"V3 std":>10} {"V3 range":>20}  |  {"V4 mean":>10} {"V4 std":>10} {"V4 range":>20} {"Status"}')
print('-' * 130)
for f in pace_feats:
    v3_str = ''
    v4_str = ''
    status = ''
    if f in v3.columns:
        s3 = v3[f]
        v3_str = f'{s3.mean():>10.2f} {s3.std():>10.3f} {f"[{s3.min():.1f}, {s3.max():.1f}]":>20}'
    else:
        v3_str = f'{"N/A":>10} {"":>10} {"":>20}'
    
    if f in v4.columns:
        s4 = v4[f]
        v4_str = f'{s4.mean():>10.2f} {s4.std():>10.3f} {f"[{s4.min():.1f}, {s4.max():.1f}]":>20}'
        if s4.std() == 0:
            status = 'DEAD'
    else:
        v4_str = f'{"DROPPED":>10} {"":>10} {"":>20}'
        status = 'DROPPED'
    
    print(f'{f:<30} {v3_str}  |  {v4_str} {status}')

## 7. Book Features Comparison

In [ ]:
book_feats = [c for c in v3.columns if 'deviation' in c or 'book' in c or 'line_spread' in c 
              or 'consensus' in c or 'softest' in c or 'hardest' in c]
book_feats = sorted(set(book_feats))

print(f'{"Feature":<30} {"V3 mean":>10} {"V3 std":>8} {"V3 nz%":>7}  |  {"V4 mean":>10} {"V4 std":>8} {"V4 nz%":>7}')
print('-' * 95)
for f in book_feats:
    v3_part = ''
    v4_part = ''
    if f in v3.columns and np.issubdtype(v3[f].dtype, np.number):
        s3 = v3[f]
        v3_part = f'{s3.mean():>10.3f} {s3.std():>8.3f} {(s3!=0).mean()*100:>6.1f}%'
    else:
        v3_part = f'{"N/A":>10} {"":>8} {"":>7}'
    if f in v4.columns and np.issubdtype(v4[f].dtype, np.number):
        s4 = v4[f]
        v4_part = f'{s4.mean():>10.3f} {s4.std():>8.3f} {(s4!=0).mean()*100:>6.1f}%'
    else:
        v4_part = f'{"DROPPED":>10} {"":>8} {"":>7}'
    print(f'{f:<30} {v3_part}  |  {v4_part}')

## 8. Prop History Features Comparison

In [ ]:
prop_feats = sorted([c for c in v3.columns if c.startswith('prop_')])

print(f'{"Feature":<30} {"V3 mean":>10} {"V3 std":>8} {"V3 nz%":>7}  |  {"V4 mean":>10} {"V4 std":>8} {"V4 nz%":>7}')
print('-' * 95)
for f in prop_feats:
    s3 = v3[f] if f in v3.columns else None
    s4 = v4[f] if f in v4.columns else None
    v3_part = f'{s3.mean():>10.3f} {s3.std():>8.3f} {(s3!=0).mean()*100:>6.1f}%' if s3 is not None else f'{"N/A":>10} {"":>8} {"":>7}'
    v4_part = f'{s4.mean():>10.3f} {s4.std():>8.3f} {(s4!=0).mean()*100:>6.1f}%' if s4 is not None else f'{"N/A":>10} {"":>8} {"":>7}'
    print(f'{f:<30} {v3_part}  |  {v4_part}')

## 9. Distribution Plots — Key Regressions

In [ ]:
# Plot distributions for key regressed features
regression_feats = ['team_pace', 'opp_def_factor', 'opp_positional_def', 
                    'position_matchup_advantage', 'starter_ratio',
                    'h2h_avg_points', 'h2h_trend_points', 'h2h_games',
                    'line_source_reliability', 'softest_book_line_bias',
                    'days_rest', 'days_since_last_30pt_game']

# Only plot features that exist in both
plot_feats = [f for f in regression_feats if f in v3.columns and f in v4.columns]
n_plots = len(plot_feats)
n_cols = 3
n_rows = (n_plots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, f in enumerate(plot_feats):
    ax = axes[i]
    s3 = v3[f].dropna()
    s4 = v4[f].dropna()
    
    # Clip extreme outliers for readability
    p1, p99 = np.percentile(pd.concat([s3, s4]), [1, 99])
    if p1 != p99:
        bins = np.linspace(p1, p99, 50)
        ax.hist(s3.clip(p1, p99), bins=bins, alpha=0.5, label=f'V3 (n={len(s3):,})', density=True)
        ax.hist(s4.clip(p1, p99), bins=bins, alpha=0.5, label=f'V4 (n={len(s4):,})', density=True)
    else:
        ax.bar(['V3', 'V4'], [s3.mean(), s4.mean()], alpha=0.7)
    
    ax.set_title(f, fontsize=10)
    ax.legend(fontsize=7)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('V3 vs V4 Feature Distributions — Key Regressions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. V4-Only Features (New in V4)

In [ ]:
v4_new = [c for c in v4.columns if c not in v3.columns and c not in meta]
print(f'{len(v4_new)} features in V4 not in V3:\n')

print(f'{"Feature":<45} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10} {"NonZero%":>8} {"CorrLabel":>10}')
print('-' * 110)
for f in sorted(v4_new):
    if f in v4.columns and np.issubdtype(v4[f].dtype, np.number):
        s = v4[f]
        corr = s.corr(v4['label']) if s.std() > 0 else 0
        status = ' DEAD' if s.std() == 0 else ''
        print(f'{f:<45} {s.mean():>10.3f} {s.std():>10.3f} {s.min():>10.2f} {s.max():>10.2f} {(s!=0).mean()*100:>7.1f}% {corr:>+10.4f}{status}')

## 11. Summary — What Needs Fixing

In [ ]:
print('=' * 80)
print('SUMMARY: V3 → V4 Dataset Issues')
print('=' * 80)
print()

# Count regressions
v4_dead_feats = r4[r4['dead']]['feature'].tolist()
v3_alive_now_dead = [f for f in v4_dead_feats if f in r3['feature'].values and not r3[r3['feature']==f].iloc[0]['dead']]
print(f'Features that were ALIVE in V3 but DEAD in V4: {len(v3_alive_now_dead)}')
for f in v3_alive_now_dead:
    v3_row = r3[r3['feature']==f].iloc[0]
    print(f'  {f}: V3 std={v3_row["std"]}, V3 range=[{v3_row["min"]}, {v3_row["max"]}]')

print()
print(f'V4-only features (new): {len(v4_new)}')
v4_new_dead = [f for f in v4_new if f in v4.columns and v4[f].std() == 0]
v4_new_alive = [f for f in v4_new if f in v4.columns and v4[f].std() > 0]
print(f'  Alive: {len(v4_new_alive)}')
print(f'  Dead: {len(v4_new_dead)} — {v4_new_dead}')

print()
v3_only_feats = [c for c in v3.columns if c not in v4.columns and c not in meta and np.issubdtype(v3[c].dtype, np.number)]
print(f'V3 features dropped from V4: {len(v3_only_feats)}')
for f in sorted(v3_only_feats):
    s = v3[f]
    print(f'  {f}: std={s.std():.3f}, nonzero={((s!=0).mean()*100):.1f}%')